# Baidu Netdisk → Google Drive (via bypy)

Download a single file from Baidu Netdisk (百度网盘) to Google Drive via Google Colab.
Uses [bypy](https://github.com/houtianze/bypy) — pure-Python, OAuth-based (no long-lived cookie to leak).

**Sandbox limitation:** `bypy` can only access files under `/我的应用数据/bypy/` on your Baidu drive. The prerequisite step below moves your target file there.

**Heads up:** Baidu throttles non-SVIP accounts to ~50–300 KB/s. Multi-GB downloads can take hours.

## Prerequisite — do this in your browser before running the notebook

1. Log into <https://pan.baidu.com> in any browser.
2. Navigate to `我的应用数据/bypy/` (create it if it doesn't exist — `我的应用数据` is shown in the left sidebar; `bypy` is a subfolder you may need to create on first use).
3. **Move** (not copy — saves your quota) your target `.zip` into that folder.
4. Note the exact filename — you'll paste it into the config cell below.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install bypy

Pure-Python install — no native wheel build, so this works on any Colab Python version.

In [ ]:
!pip install -q bypy

## 3. Authorize bypy (OAuth, one-time per Colab VM)

Running the next cell will:
1. Print a Baidu authorization URL — **open it in a new browser tab**.
2. Log into your Baidu account if asked, then click **授权 / Authorize** for the "bypy" app.
3. Baidu shows you a short authorization code — **copy it**.
4. Paste the code into the input box that appears at the top of Colab, then press Enter.

On success the cell prints a `Quota:` line showing your drive usage.

> The code is short-lived (expires in minutes). If it fails, re-run this cell and use a freshly-generated one. There is no long-lived secret stored in the notebook — bypy keeps a refresh token in `~/.bypy/` on the Colab VM, which disappears when the VM resets.

In [ ]:
!bypy info

## 4. List the bypy sandbox to confirm your file is there

This shows the contents of `/我的应用数据/bypy/` on your Baidu drive. Your target `.zip` should be in the list.

In [ ]:
!bypy list

## 5. Configure source and destination

- `BAIDU_FILE` — the filename as it appears in `bypy list`. Just the name, no leading slash, no folder path.
- `DRIVE_DEST` — the Drive folder to save into. Created if missing.

In [ ]:
import os

BAIDU_FILE = 'data.zip'                                   # ← edit
DRIVE_DEST = '/content/drive/MyDrive/AutoLLM_data/raw'    # ← edit

LOCAL_PATH = os.path.join(DRIVE_DEST, BAIDU_FILE)
os.makedirs(DRIVE_DEST, exist_ok=True)
print('From:', BAIDU_FILE)
print('To:  ', LOCAL_PATH)

## 6. Download

`bypy downfile <remote_name> <local_path>` downloads one file. The `-v` flag prints progress.

**Resumable**: if Colab disconnects mid-download or you re-run this cell, bypy resumes from where it left off (skips already-downloaded bytes).

In [ ]:
!bypy downfile "{BAIDU_FILE}" "{LOCAL_PATH}" -v

## 7. Verify

In [ ]:
!ls -la "{LOCAL_PATH}"
!du -h "{LOCAL_PATH}"

## Troubleshooting

- **"Authorization code rejected" / OAuth fails** — codes expire in a few minutes. Re-run the auth cell (`bypy info`) and paste a *freshly-generated* code from a fresh visit to the Baidu URL.
- **`Error: File not found` on download** — `bypy` only sees `/我的应用数据/bypy/`. Run cell 10 (`bypy list`) to confirm the file is in the sandbox. If you saved it elsewhere on your drive, move it via the Baidu web UI.
- **Very slow download (<100 KB/s)** — Baidu's non-SVIP throttle. The only real fix is Baidu SVIP (超级会员) for the month. Otherwise, accept the speed and let Colab run overnight.
- **Colab disconnects mid-download** — free Colab kills idle sessions after ~90 min and caps total runtime at ~12 h. Re-run the download cell — bypy resumes from where it stopped.
- **Google Drive runs out of space** — free Drive is 15 GB shared with Gmail and Google Photos. Check current usage at <https://drive.google.com/storage> before starting a multi-GB download. To get more, subscribe to **Google One**: 100 GB ($2/mo), 200 GB ($3/mo), 2 TB ($10/mo), 5 TB, etc. No free way to exceed 15 GB.
- **`bypy: command not found`** — pip install path issue. Use `!python -m bypy ...` instead.

### Test with a small file first

Before kicking off the multi-GB zip, upload a few-MB test file into `/我的应用数据/bypy/` and run cells 8 → 10 → 12 → 14 → 16 against that. Confirms the full pipeline works without committing to hours of throttled bandwidth on a wrong filename or path.